<h1>Trabalho de IA</h1>
<br>
<h2>Grupo:</h2>
<ul>
    <li>Diogo Alexandre Alves da Silva - nº 31504</li>
    <li>Gonçalo Rodrigues - nº 31507</li>
    <li>Rui Barbosa - nº 31515</li>
</ul>

<h2>Introdução</h2>
<p>
Este trabalho foi desenvolvido no âmbito da unidade curricular de Inteligência Artificial (2026-27) da licenciatura em Engenharia de Sistemas Informáticos (UPCA). O tema é Comida Local: uma plataforma digital que liga produtores locais, clientes e centros de recolha (armazéns), onde uma frota de veículos recolhe os produtos junto dos produtores e os entrega nos locais escolhidos pelos clientes.
</p>
<h3>Motivação</h3>
<p>
A crescente procura de comida local na UE faz com que os pequenos produtores enfrentem custos logísticos elevados e preços voláteis. Cadeias de abastecimento curtas reduzem emissões de transporte e desperdício alimentar, mas tornam a logística o principal desafio: é preciso escolher rotas eficientes de recolha e entrega que respeitem as janelas de tempo de cada pedido e a capacidade dos veículos.
</p>
<h3>Objetivo</h3>
<p>O objetivo é desenvolver um agente inteligente que planeie essas rotas, aplicando e comparando três tipos de algoritmos:</p>
<ul>
    <li>Pesquisa informada (Greedy e A*)</li>
    <li>Pesquisa local (Hill Climbing)</li>
    <li>Problemas de Satisfação de Restrições (CSP)</li>
</ul>
<p>O trabalho é feito em Python e Jupyter Notebook, sobre um dataset que cobre o distrito de Braga, incluindo a Póvoa de Varzim.</p>
<h3>Estrutura</h3>
<ol>
    <li><b>Introdução:</b> contexto, motivação e objetivo.</li>
    <li><b>Resumo EDA:</b> principais conclusões da análise exploratória (<code>EDA.ipynb</code>).</li>
    <li><b>Problema e abordagem:</b> terminologia, pressupostos, restrições, modelação, função objetivo e algoritmos.</li>
    <li><b>Implementação:</b> código e execução dos algoritmos.</li>
    <li><b>Resultados:</b> melhores rotas, tempos de execução e comparação entre algoritmos.</li>
    <li><b>Conclusão:</b> balanço e melhorias futuras.</li>
</ol>


<h2>Resumo EDA</h2>
<p>
O dataset (versão <i>tiny</i>) descreve 5 produtores, 1 centro de recolha, 2 veículos, 8 clientes, 6 locais de entrega, 10 encomendas com 23 pedidos (cada um com janelas de tempo) e uma matriz com os 132 pares de distâncias e tempos de viagem entre os 12 pontos de rota. A análise completa está em <code>EDA.ipynb</code>.
</p>
<p>
A análise exploratória teve como objetivos: (i) validar a integridade dos dados e a coerência das janelas de tempo (<code>availability_start/end</code> no produtor e <code>delivery_start/end</code> no cliente); (ii) comparar pesos das encomendas com as capacidades dos veículos; (iii) visualizar a distribuição geográfica das paragens. A matriz de distâncias é uma aproximação sintética e simétrica: <code>distance_km</code> é a distância em linha reta multiplicada por um fator de estrada (1.3 por defeito) e <code>duration_min</code> assume uma velocidade média de 45 km/h.
</p>
<p><b>Principais conclusões:</b></p>
<ul>
    <li><b>Integridade:</b> os dados são consistentes, sem valores em falta, duplicados ou referências inválidas entre tabelas.</li>
    <li><b>Janelas de tempo:</b> todas válidas. Considerando o percurso produtor → centro de recolha → cliente, 22 dos 23 pedidos são viáveis. O pedido 8 (encomenda 3) falha por cerca de 28 minutos (48 minutos com os 10 min de serviço por paragem usados no modelo), pelo que será entregue fora da janela. As janelas de entrega são curtas (1 a 4 horas) e são a restrição mais apertada.</li>
    <li><b>Capacidade:</b> a carga total (~249 kg) é cerca de 10% da capacidade da frota (2454.5 kg) e ~25% do veículo mais pequeno, pelo que a capacidade não é limitativa e o problema é dominado pelas janelas de tempo.</li>
    <li><b>Matriz de distâncias:</b> completa e simétrica, com distância em linha reta × 1.3 e velocidade de 45 km/h. Respeita a desigualdade triangular, pelo que a distância em linha reta é uma heurística admissível para o A*.</li>
    <li><b>Estrutura temporal:</b> os pedidos distribuem-se por 13 dias e formam 8 grupos temporais independentes (o maior com 8 pedidos), que podem ser resolvidos separadamente.</li>
    <li><b>Incoerências a tratar:</b> <code>order_datetime</code> é posterior às janelas na maioria dos pedidos (não é usado como restrição); os veículos indicam ~58-60 km/h contra os 45 km/h da matriz (usa-se a matriz); os produtores 1 e 5 partilham as mesmas coordenadas.</li>
</ul>

<h2>Problema e abordagem</h2>
<h3>Terminologia</h3>
<p>
Uma <b>encomenda</b> (10 no dataset) é composta por um ou mais <b>pedidos</b> (23 no total). Um pedido corresponde a uma linha da tabela <code>order_items</code>: um produto de um produtor, com a respetiva quantidade, janela de recolha e janela de entrega. O pedido é a unidade de planeamento e é representado pela classe <code>Request</code>.
</p>

<h3>Definição do problema</h3>
<p>
Dado um conjunto de pedidos (cada um com local de recolha no produtor, local de entrega no cliente e as respetivas janelas de tempo) e uma frota de veículos com capacidade limitada, determinar as rotas que servem todos os pedidos com o menor custo, cumprindo as restrições.
</p>

<p><b>Pressupostos.</b></p>
<ul>
    <li>Cada veículo parte e termina no centro de recolha a que pertence (<code>center_id</code>).</li>
    <li>O tempo é absoluto (data e hora). Se um veículo chega antes do início de uma janela, espera.</li>
    <li>Os tempos e distâncias de viagem vêm da matriz (45 km/h). Cada paragem (recolha, descarga, carga ou entrega) demora 10 min de serviço (<code>SERVICE_MIN</code>). O armazém não tem limite de capacidade nem de tempo de espera dos produtos.</li>
    <li>A <code>order_datetime</code> não é usada como restrição (ver EDA).</li>
</ul>
<p><b>Restrições obrigatórias (<i>hard</i>).</b></p>
<ul>
    <li>A recolha ocorre dentro da janela de disponibilidade do produto no produtor, isto é, não antes de <code>availability_start</code> e não depois de <code>availability_end</code>.</li>
    <li>A entrega ocorre a partir do início da janela de entrega do cliente (<code>delivery_start</code>).</li>
    <li>A carga transportada não pode exceder a capacidade do veículo.</li>
    <li>Cada pedido é recolhido e entregue exatamente uma vez.</li>
    <li>Todo o produto passa pelo centro de recolha (armazém): é recolhido no produtor, entregue no armazém e só depois seguido para o cliente.</li>
    <li>Um veículo pode recolher no armazém vários pedidos de uma só vez e entregá-los aos clientes no mesmo percurso, mas só pode sair com produtos que já tenham chegado ao armazém.</li>
</ul>
<p><b>Restrição desejável (<i>soft</i>).</b></p>
<ul>
    <li>A entrega deve ocorrer até ao fim da janela de entrega (<code>delivery_end</code>). Esta restrição pode ser violada com uma penalização, porque o EDA mostra que o pedido 8 não a consegue cumprir, mesmo no melhor caso (tem um atraso mínimo inevitável, mesmo no melhor caso).</li>
</ul>
<h3>Modelação</h3>
<p>
O problema é representado pelas classes <code>Request</code> (pedido), <code>Vehicle</code> (veículo) e <code>State</code> (estado: posição, tempo atual, carga transportada, produtos no armazém e pedidos pendentes/concluídos), com uma função de custo. Cada pedido tem duas fases: produtor → armazém e armazém → cliente.
</p>
<p>
<b>Estado.</b> Para cada veículo guarda-se (localização, tempo atual, carga em kg); para cada pedido, a fase em que está (pendente, recolhido e a caminho do armazém, no armazém, a bordo a caminho do cliente ou entregue) e o instante em que chegou ao armazém. No estado inicial todos os veículos estão no centro, no tempo 0 e vazios, e todos os pedidos estão pendentes.
</p>
<p>
<b>Ações</b> (de um veículo): <code>P r</code> recolhe o pedido r no produtor; <code>C</code> descarrega no armazém o que o veículo traz; <code>L r</code> vai ao armazém, descarrega e carrega o pedido r; <code>D r</code> entrega r ao cliente. Cada ação leva o veículo ao local respetivo (tempo e distância da matriz), espera se chegar antes do início de uma janela e é rejeitada se violar uma restrição obrigatória. Se <code>L r</code> depende de um pedido que outro veículo ainda não levou ao armazém, a ação fica bloqueada até esse pedido lá chegar.
</p>
<p>
<b>Solução, objetivo e função de custo.</b> Uma solução é uma lista de ações por veículo. O estado objetivo é aquele em que todos os pedidos estão entregues, e os veículos regressam depois ao centro. O custo a minimizar é
</p>
<p style="text-align:center"><code>custo = distância total (km) + 10 × minutos de atraso face a delivery_end</code></p>
<p>
incluindo o regresso ao centro. Com a penalização de 10 km por minuto, atrasar só compensa quando o atraso é inevitável, como no pedido 8.
</p>
<h3>Algoritmos</h3>
<p>
<b>Pesquisa informada (Greedy e A*).</b> Pesquisa no espaço de estados com a fila de prioridade ordenada por f = h (Greedy) ou f = g + h (A*), em que g é o custo acumulado. Foram usadas duas heurísticas, ambas somadas ao atraso inevitável (parte <i>soft</i>): a <b>h1</b> é a distância da cadeia produtor → centro → cliente do pedido mais exigente; a <b>h2</b> é o máximo entre a h1 e a soma dos troços "com carga" de pedidos cujas janelas não se sobrepõem (que nunca viajam juntos), calculada por programação dinâmica. A h1 é admissível. A h2 é admissível enquanto nenhum pedido se atrasa, mas pode sobrestimar quando um pedido é entregue depois de <code>delivery_end</code> e passa a partilhar viagem com o seguinte (ver Limitações). Cortam-se estados sem saída (algum pedido já não consegue ser recolhido a tempo) e estados dominados.
</p>
<p>
<b>Pesquisa local (Hill Climbing).</b> Parte de uma solução completa e, por primeira melhoria, aceita o primeiro vizinho viável com custo menor, parando num ótimo local. Os vizinhos são: trocar duas ações, mover uma ação para outra posição ou veículo, mover a cadeia P-L-D de um pedido para outro veículo ou posição, e remover uma visita ao centro. Como só reorganiza pedidos que a solução já serve, precisa de uma solução inicial que sirva todos.
</p>
<p>
<b>CSP.</b> Cada pedido dá quatro variáveis, uma por paragem: <b>P</b> (recolha no produtor), <b>C</b> (descarga no armazém do veículo que recolheu), <b>L</b> (carga no armazém pelo veículo que vai entregar) e <b>D</b> (entrega ao cliente). O domínio de cada uma é um par (veículo, hora de início do serviço), com a hora em múltiplos de 5 min. A descarga C tem ainda o valor "virtual" (veículo, sem hora), usado quando o mesmo veículo recolhe e entrega e a descarga é feita pela própria carga L. O veículo que recolhe pode assim ser diferente do que entrega, como no modelo de estados.
</p>
<p>
<i>Restrições unárias:</i> janela de recolha, entrega não antes de <code>delivery_start</code> e capacidade do veículo. <i>Binárias:</i> P e C no mesmo veículo, com C depois da viagem; L só depois de os produtos chegarem ao armazém (se a descarga é virtual, L é no mesmo veículo da recolha); L e D no mesmo veículo, com D depois da viagem; e, no mesmo veículo, duas paragens não se sobrepõem (serviço mais deslocação entre os locais). <i>Global:</i> a carga a bordo não excede a capacidade (limite conservador, de P até D). <i>Soft:</i> o domínio da entrega só admite inicialmente o atraso inevitável de cada pedido e é alargado por degraus (0, 15, 30, 60, 120, 480 e 1440 min) até haver solução; o custo penaliza o atraso.
</p>
<p>
Resolve-se por <i>backtracking</i> com MRV, <i>forward checking</i>, ordenação dos valores pelo menor custo extra (heurística para a restrição <i>soft</i>) e reinícios aleatórios contra o <i>thrashing</i>. As viagens com vários pedidos não são impostas nem corrigidas depois: surgem quando paragens de pedidos diferentes ficam seguidas no mesmo veículo, porque o custo é a distância entre paragens consecutivas. O custo final é sempre recalculado por <code>simulate</code>, que executa cada rota o mais cedo possível, por isso é menor ou igual ao custo planeado sobre a grelha de tempo.
</p>

<h2>Implementação</h2>

<p>O código está separado em dois módulos, importados neste notebook:</p>
<ul>
    <li><code>model.py</code>: modelo do problema (<code>Request</code>, <code>Vehicle</code>, <code>State</code>, <code>Problem</code>), ações, função de custo, simulador de rotas, verificador de viabilidade (<code>audit</code>) e leitura dos dados (<code>load_problem</code>).</li>
    <li><code>algoritmos.py</code>: Greedy e A* (<code>best_first</code>, heurísticas <code>h_chain</code> e <code>h_disjoint</code>), Hill Climbing (<code>hill_climbing</code>), CSP (<code>csp_solve</code>, <code>csp_complete</code>) e as funções que preparam o problema (<code>temporal_groups</code>, <code>initial_solution</code>).</li>
</ul>
<p><b>Parâmetros usados.</b></p>
<ul>
    <li><b>Greedy e A*:</b> limite de 200 000 nós e 60 s por pesquisa. O Greedy usa a h1; o A* é testado com a h1 e com a h2.</li>
    <li><b>Hill Climbing:</b> como pede o guia, parte da solução do <b>Greedy</b>. Como termo de comparação, corre também a partir de <code>initial_solution</code> (A* e, se não terminar, A* ponderado com pesos 2, 5 e 20), que é a solução inicial usada pelo agente completo (<code>solve_all</code>). Usa semente 0 e um máximo de 500 iterações.</li>
    <li><b>CSP:</b> MRV, <i>forward checking</i> e valores por custo extra (<code>"soft"</code>), em modo de primeira solução, com 12 reinícios aleatórios. <code>csp_complete</code> repete a pesquisa alargando a folga de atraso dos domínios (0, 15, 30, 60, 120, 480 e 1440 min) até haver solução. Com <code>optimize=True</code> continua em <i>branch &amp; bound</i> e devolve <code>complete</code> (a árvore foi esgotada) e <code>timed_out</code>.</li>
</ul>
<p>Os pedidos distribuem-se por 13 dias e formam grupos que não interferem entre si (ver EDA), por isso o problema é resolvido por <b>grupos temporais</b> (<code>temporal_groups</code>) e as rotas são juntas no fim.</p>
<h3>Carregar o problema</h3>


In [1]:
import time
import pandas as pd
from model import load_problem
from algoritmos import (solve_all, temporal_groups, best_first, h_chain, h_disjoint,
                        initial_solution, hill_climbing, csp_solve, csp_complete)

P, origin = load_problem("data_tiny")
print(f"{P.n} pedidos, {P.K} veículos, carga total {sum(P.w):.1f} kg")

23 pedidos, 2 veículos, carga total 249.0 kg


<h3>Agente completo</h3>
<p><code>solve_all</code> resolve cada grupo (A* e, se preciso, A* ponderado), melhora-o com Hill Climbing, junta as rotas de todos os grupos e valida o conjunto com <code>audit</code>.</p>

In [ ]:
t0 = time.time()
routes, info, audit = solve_all(P)
print(f"Tempo total: {time.time() - t0:.1f} s")
for g in info:
    print(g)
print(audit)

<h3>Comparação dos algoritmos por grupo</h3>
<p>Cada algoritmo é corrido isoladamente em cada grupo, com os mesmos limites (200 000 nós, 60 s). O Hill Climbing corre a partir da solução do Greedy (<code>HC (Greedy)</code>, a pedida pelo guia) e a partir de <code>initial_solution</code> (<code>HC (A*)</code>); o tempo do HC não inclui o da solução inicial. O CSP corre em modo de primeira solução (<code>optimize=False</code>). Guardam-se o custo, os km, o tempo, os nós expandidos (avaliações de vizinhos, no Hill Climbing) e, no CSP, os <i>backtracks</i>. Guarda-se também o histórico de custo de cada Hill Climbing, para as curvas de custo por iteração.</p>


In [ ]:
from algoritmos import plan_to_routes

rows, hc_hist = [], {}
for i, g in enumerate(temporal_groups(P), 1):
    Q = P.subset([P.req[r].id for r in g])

    for nome, kw in [("Greedy (h1)", dict(h=h_chain, mode="greedy")),
                     ("A* (h1)", dict(h=h_chain, mode="astar")),
                     ("A* (h2)", dict(h=h_disjoint, mode="astar"))]:
        r = best_first(Q, max_nodes=200000, time_limit=60, **kw)
        rows.append(dict(grupo=i, pedidos=Q.n, algoritmo=nome, encontrou=r["found"], custo=r["cost"], km=r["km"],
                         segundos=round(r["seconds"], 2), nos=r["expanded"], backtracks=None))
        if nome == "Greedy (h1)":
            greedy = r

    # Hill Climbing a partir do Greedy (pedido no guia) e a partir de initial_solution (A* / A* ponderado)
    inicios = {}
    if greedy["found"]:
        inicios["HC (Greedy)"] = plan_to_routes(Q, greedy["plan"])
    rt, _ = initial_solution(Q)
    if rt:
        inicios["HC (A*)"] = rt
    for nome, rt0 in inicios.items():
        hc = hill_climbing(Q, rt0)
        hc_hist[(i, nome)] = hc["history"]
        rows.append(dict(grupo=i, pedidos=Q.n, algoritmo=nome, encontrou=True, custo=round(hc["cost"], 2),
                         km=round(Q.simulate(hc["routes"])["km"], 2), segundos=round(hc["seconds"], 2), nos=hc["evals"], backtracks=None))

    c = csp_complete(Q, var_order="mrv", value_order="soft", forward_checking=True, time_limit=60)
    rows.append(dict(grupo=i, pedidos=Q.n, algoritmo="CSP (MRV+FC)", encontrou=c["found"], custo=c["cost"], km=c.get("km"),
                     segundos=round(c["seconds"], 2), nos=c["nodes"], backtracks=c["backtracks"]))

df = pd.DataFrame(rows)
df


<h3>Hill Climbing: custo por iteração</h3>
<p>Curvas de custo por iteração do Hill Climbing em cada grupo com mais de um pedido, a partir do Greedy (linha contínua) e a partir do A* (tracejado). Cada iteração aceita o primeiro vizinho que melhora o custo; a curva fica plana quando se atinge um ótimo local.</p>


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# só os grupos com mais de um pedido (nos outros a solução inicial já é ótima e a curva é plana)
grupos_hc = sorted(g for g in df.grupo.unique() if df[df.grupo == g].pedidos.iloc[0] > 1)
fig, axs = plt.subplots(1, len(grupos_hc), figsize=(4.2 * len(grupos_hc), 3.4), squeeze=False)
for ax, g in zip(axs[0], grupos_hc):
    for nome, estilo in (("HC (Greedy)", "-"), ("HC (A*)", "--")):
        h = hc_hist.get((g, nome))
        if h:
            ax.plot(range(len(h)), h, estilo, marker="o", ms=4, label=nome)
    ax.set(title=f"Grupo {g} ({df[df.grupo == g].pedidos.iloc[0]} pedidos)", xlabel="iteração", ylabel="custo")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True)); ax.grid(alpha=.3)
axs[0][0].legend()
plt.tight_layout(); plt.show()


<h3>CSP: atribuição viável</h3>
<p>Exemplo de uma atribuição viável encontrada pelo CSP (MRV + FC, primeira solução), no grupo 3. Cada variável recebe um par (veículo, hora); a tabela mostra as paragens resultantes por ordem, depois de o simulador executar a rota o mais cedo possível, e indica o número de nós e de <i>backtracks</i> da pesquisa.</p>


In [ ]:
g3 = temporal_groups(P)[2]
Q3 = P.subset([P.req[r].id for r in g3])
c3 = csp_complete(Q3, var_order="mrv", value_order="soft", forward_checking=True, time_limit=60)
print(f"Pedidos {[q.id for q in Q3.req]} | nós: {c3['nodes']} | backtracks: {c3['backtracks']} | custo: {c3['cost']:.2f} | auditoria ok: {Q3.audit(c3['routes'])['ok']}")
Q3.timeline(c3["routes"], origin)


<h2>Resultados</h2>
<p><i>[preencher: melhores rotas por veículo, tempos de execução e comparação entre algoritmos]</i></p>

<h2>Conclusão</h2>
<p><i>[preencher depois dos resultados: balanço dos resultados, do desenvolvimento e das ferramentas, e melhorias futuras do agente]</i></p>